# ADK Mastery: Dynamic Agent Skills Demo (February 2026 Preview)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maruti123/partner-demos/blob/main/partner-demos-feb-2026/adk_skills_mastery_demo.ipynb)

This notebook demonstrates the **Skills** feature in the Google **Agent Development Kit (ADK)**. Skills allow agents to load complex instructions and domain-specific resources on-demand.

## Use Case
A general-purpose developer assistant needs to perform a specialized 'Code Review'. Instead of having 50 pages of instructions in its prompt at all times, it dynamically loads the 'Code Reviewer' skill only when needed.

### Requirements
- `google-adk >= 1.26.0` installed.
- A directory-based skill with a valid `SKILL.md` file.

In [ ]:
# 1. Setup and Environment
!pip install google-adk google-genai nest-asyncio
from google.colab import auth
auth.authenticate_user()

import os
import pathlib
import asyncio
import nest_asyncio
from google.adk import Agent, Runner
from google.adk.skills import load_skill_from_dir
from google.adk.tools.skill_toolset import SkillToolset
from google.adk.sessions.in_memory_session_service import InMemorySessionService
from google.genai import types

nest_asyncio.apply()
project_id = 'YOUR_PROJECT_ID'  # @param {type:"string"}

### 2. [PREREQUISITES] Create a Local Skill Directory

We create a folder named 'code-reviewer' and a `SKILL.md` file.

In [ ]:
skill_root = pathlib.Path("./code-reviewer")
skill_root.mkdir(parents=True, exist_ok=True)

skill_md_content = """---
name: "code-reviewer"
description: "Specialized skill for performing technical code audits."
---

# Code Reviewer Instructions
1. Scan code for anti-patterns (e.g., hardcoded keys).
2. Provide a structured feedback report.
"""

with open(skill_root / "SKILL.md", "w") as f:
    f.write(skill_md_content)

print(f"Skill created at: {skill_root}")

### 3. Load the Skill and Initialize the Agent

We use **Gemini 3.1 Pro** and route to Vertex AI via environment variables.

In [ ]:
# Set environment variables to route ADK to Vertex AI
import os
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = project_id
os.environ["GOOGLE_CLOUD_LOCATION"] = "us-central1"

reviewer_skill = load_skill_from_dir(str(skill_root))
skill_toolset = SkillToolset(skills=[reviewer_skill])

agent = Agent(
    model="gemini-3.1-pro",
    name="DevAssistant",
    instruction="You are a helpful software engineer. Use your skills to assist the user.",
    tools=[skill_toolset]
)

runner = Runner(
    agent=agent, 
    session_service=InMemorySessionService(),
    app_name="SkillDemoApp",
    auto_create_session=True
)
print("Agent and SkillToolset ready (using Vertex AI via environment variables).")

### 4. Execute the Skill-Driven Workflow

We trigger the agent with a code snippet. It will automatically load the 'code-reviewer' skill.

In [ ]:
user_query = "Please review this code for me: def calc(x, y): k='AIza'; return x+y"
print(f"User: {user_query}\n")

async def run_demo():
    # Correctly construct the Content object for ADK
    message = types.Content(parts=[types.Part(text=user_query)], role='user')
    
    response_stream = runner.run_async(user_id="lead_qa", session_id="s1", new_message=message)
    
    print("--- Agent Execution ---")
    async for event in response_stream:
        if event.content and event.content.parts:
            text = event.content.parts[0].text
            if text: print(f"Agent: {text}")
        if hasattr(event, 'function_call') and event.function_call:
             print(f"[SYSTEM]: Calling Skill Tool '{event.function_call.name}'")

asyncio.run(run_demo())

### 5. Things to remember or know
- **Skill Loading**: Use `load_skill_from_dir` to dynamically inject specialized logic (like the `code-reviewer` skill) into your agent sessions.
- **Modern Routing**: ADK 2026 uses the `Agent` class which abstracts the underlying `GOOGLE_GENAI_USE_VERTEXAI` complexity.
- **Gemini 3.1 Pro**: The flagship model for high-fidelity reasoning and precise skill execution.